# Path 2 - HuggingFace
HuggingFace (HF) is a free platform where user can upload models (of various kinds, not just LLMs) that can then be used through their `transformers` library. To be able to use the models on HF you don't need to create an account, however, some models are 'gated' and require approval from the creator before being able to use them (it is the case e.g. for Llama models). For those models, it's required both authentication and authorization to use the model.

### 1. First simple generation
For the means of this lab, we will use the model `Qwen/Qwen3.5-2B`, which is a non-gated fairly small model that, besides text, also support images and videos. For the assignment and the project you can choose the model that you prefer from the [HF catalogue](https://huggingface.co/models).

Popular alternatives are:
* `Qwen/Qwen3.5-0.6B`, a smaller variant in case you cannot fit the above one in the memory on your local machine
* `google/gemma-4-E4B-it`, a different multimodal model by google
* `google/gemma-4-E2B-it`, a smaller variant of the above

If you have a machine with a GPU or unified memory (e.g., Apple Silicon), you can choose the biggest model that can fit in the VRAM or RAM, respectively. To see an estimate of how much memory a specific model requires you can use the commands below. It it generally advisable to leave at least 10% of memory free.

If you are running the model on CPU, you might want to reduce the model size further as this will speed up inference times.

In [4]:
%%bash

export TRANSFORMERS_VERBOSITY=error

# This will print a table with the estimated memory usage for the model. 
# The column "Total size" shows the total memory usage for the model in GB when varying on model precision.
# Lower precision will use less memory at the cost of degradation in performance.
# It is common to load language models in half precision (e.g., float16 or bfloat16).
# You can also use int8 or int4 quantization to reduce memory usage even further.
accelerate estimate-memory "Qwen/Qwen3-2B"

Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/bin/accelerate", line 3, in <module>
    from accelerate.commands.accelerate_cli import main
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/accelerate/__init__.py", line 16, in <module>
    from .accelerator import Accelerator
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/accelerate/accelerator.py", line 35, in <module>
    from accelerate.utils.dataclasses import FP8BackendType
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/accelerate/utils/__init__.py", line 14, in <module>
    from ..parallelism_config import ParallelismConfig
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/accelerate/parallelism_config.py", line 20, in <module>
    from accelerate.utils.dataclasses import (
    ...<4 lines>...
    )
  File "/Library/Frameworks/P

CalledProcessError: Command 'b'\nexport TRANSFORMERS_VERBOSITY=error\n\n# This will print a table with the estimated memory usage for the model. \n# The column "Total size" shows the total memory usage for the model in GB when varying on model precision.\n# Lower precision will use less memory at the cost of degradation in performance.\n# It is common to load language models in half precision (e.g., float16 or bfloat16).\n# You can also use int8 or int4 quantization to reduce memory usage even further.\naccelerate estimate-memory "Qwen/Qwen3-2B"\n'' returned non-zero exit status 1.

In [ ]:
from transformers import AutoModelForMultimodalLM, AutoProcessor, logging, BitsAndBytesConfig


# avoid having too much output from the transformers library
logging.set_verbosity_warning()

# fairly small but good model
MODEL_NAME = "Qwen/Qwen3-2B"

quantization_config = None

## If you need to use quantization, you can specify the quantization configuration here
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,  # set to True to use 4-bit quantization, you should use either 4-bit or 8-bit, not both
#     load_in_8bit=True,  # set to True to use 8-bit quantization, you should use either 4-bit or 8-bit, not both
# )


# We're using the `AutoModelForMultimodalLM` class to enable multimodal generation
# If you are not using a multimodal model, you can use AutoModelForCausalLM instead
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"  # automatically uses right device e.g., GPU if available
)

# We're using the `AutoProcessor` class to enable multimodal generation
# Normally, you can use AutoTokenizer
processor = AutoProcessor.from_pretrained(MODEL_NAME)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ImportError: 
AutoModelForMultimodalLM requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


#### Exercise 1

Start with using the model to predict the next part in a conversation. You need to tokenize the input, generate the response, detokenize it and print it.

In [ ]:
conversation = [
    {
        "role": "system",  # optional, could start directly with user
        "content": "You are a helpful pirate. Only reply with pirate jargon.",  # system prompt
    },
    {
        "role": "user",
        "content": "Tell me about yourself",  # user query
    },
]

# This will format the conversation in the correct format for the model
text_prompt = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    # some models support thinking and have this either enabled or disabled by default, check the model card for more information
    # enable_thinking=True
)
print("Formatted conversation: ", text_prompt)

# This will tokenize the input
inputs = processor(text=[text_prompt], return_tensors="pt")
print("Tokenized input: ", inputs)

# Make sure the input tensors are on the same device as the model (e.g. GPU)
inputs = inputs.to(model.device)

# generate response, will output tokens (NOT text)
generated_ids = model.generate(**inputs, max_new_tokens=128)
print("Generated tokens: ", generated_ids)

# This will remove the input from the generated tokens
generated_ids = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
# optional: if you have thinking enabled, you should also remove the thinking tokens from the generated tokens
# different model have different delimiters for thinking, check the model card for more information

# decode the generated tokens to text
generated_text = processor.batch_decode(generated_ids, clean_up_tokenization_spaces=False)
print("Generated text: ", generated_text[0])

### 2. Generation parameters
When asking the model to generate some text, there are different parameters that you can tune to improve on the final quality of the text. [Here](https://huggingface.co/docs/transformers/generation_strategies) is an overview of the parameters that you can change. Try some of them in different context and understand how they affect the final generated text. Feel also free to explore different decoding strategies.

#### Exercise 2

Play with the output temperature, which controls the randomness of the generated text `temperature=0` means deterministic output, while `temperature=1` means maximum randomness (try some intermediate value too) and keep the `max_new_tokens` to 50 so that the output is not too long.

In [ ]:
from transformers import GenerationConfig

low_temp_config = GenerationConfig(
    do_sample=False,  # this is equivalent to temperature=0.0
    max_new_tokens=50,
)
high_temp_config = GenerationConfig(
    do_sample=True,
    max_new_tokens=50,
    temperature=1.0,
)

generated_ids = model.generate(
    **inputs,
    generation_config=low_temp_config
)
generated_ids = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
print("Generated text with low temperature parameters: ", generated_text[0])

generated_ids = model.generate(
    **inputs,
    generation_config=high_temp_config
)
generated_ids = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
print("Generated text with high temperature parameters: ", generated_text[0])

/Users/paul/Desktop/2026_Autumn/LLM_Course/LLM-Course_MushroomChatbot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


NameError: name 'model' is not defined

#### Exercise 3

Try out different `top_k` values, which controls how many tokens the model considers for output `top_k=1` means the model considers only one token for output (the one with the highest probability) `top_k=50` means the model considers the top 50 tokens for output.

In [ ]:
low_top_k_config = GenerationConfig(
    do_sample=True,
    max_new_tokens=50,
    top_k=1,
)
high_top_k_config = GenerationConfig(
    do_sample=True,
    max_new_tokens=50,
    top_k=50,
)

generated_ids = model.generate(
    **inputs,
    generation_config=low_top_k_config
)
generated_ids = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
print("Generated text with low top_k parameters: ", generated_text[0])

generated_ids = model.generate(
    **inputs,
    generation_config=high_top_k_config
)
generated_ids = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
print("Generated text with high top_k parameters: ", generated_text[0])

#### Exercise 4

The same exercise as before but now with `top_p`, which controls how the model selects tokens for output `top_p=0.1` means the model selects tokens that make up 10% of the cumulative probability mass `top_p=0.9` means the model selects tokens that make up 90% of the cumulative probability mass `top_p` filters tokens *after* applying `top_k`.

Can you determine a rule of thumb as to how `top_k` and `top_p` affect the output results? (If you can't try to push the values to extreme values)

In [ ]:
low_top_p_config = GenerationConfig(
    do_sample=True,
    max_new_tokens=50,
    top_p=0.1,
)
high_top_p_config = GenerationConfig(
    do_sample=True,
    max_new_tokens=50,
    top_p=0.9,
)

generated_ids = model.generate(
    **inputs,
    generation_config=low_top_p_config
)
generated_ids = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
print("Generated text with low top_p parameters: ", generated_text[0])

generated_ids = model.generate(
    **inputs,
    generation_config=high_top_p_config
)
generated_ids = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
print("Generated text with high top_p parameters: ", generated_text[0])

### 3. Add images to the prompt
This model, beside text also accepts images (and videos; Gemma supports audio as well). 


#### Exercise 5
Try prompting it with one. Choose an interesting image and prompt the model with a query about it.

You can use the [library's docs](https://huggingface.co/docs/transformers/chat_templating_multimodal).

Use [PIL](https://pillow.readthedocs.io/en/stable/) to load an image. It should already be present in the Python environment.

In [ ]:
from PIL import Image

IMAGE_PATH = "./data/engineer_fitting_prosthetic_arm.jpg"

# load local image
image = Image.open(IMAGE_PATH)

conversation = [
    # we skipped the system prompt here
    {
        "role": "user",
        "content": [
            {
                "type": "image",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

# Preprocess the inputs, same as before
text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

inputs = processor( # now we add images
    text=[text_prompt], images=[image], padding=True, return_tensors="pt"
)
inputs = inputs.to(model.device)

# Inference: Generation of the output
output_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(inputs.input_ids, output_ids)
]
output_text = processor.batch_decode(
    generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
)
print(output_text[0])

### 4. Retrieval Augmented Generation (RAG)

#### Exercise 6

Depending on the application of the project, you might need to extract text from given documents and include it as additional context. This becomes especially relevant if you have many documents that cannot possibly fit into the model's context window. To more easily implement a RAG pipeline we recommend the use of one of these libraries: [LangChain](https://docs.langchain.com/oss/python/langchain/overview), [LlamaIndex](https://docs.llamaindex.ai/en/stable/examples/), [Haystack](https://docs.haystack.deepset.ai/docs/intro).

For the solution of this lab we will use *LangChain*.

It can be useful to split this exercise into these steps:
1. Read one or more documents using pdfminer or similar
2. Split the documents into small chunks
3. Get and store the embeddings for each chunks
4. Given a query, retrieve the most relevant chunk(s) and appropriately prompt your LLM

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings  # get embeddings from a HuggingFace model
from langchain_core.vectorstores import InMemoryVectorStore  # "db" to store and retrieve embeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter  # split long documents into chunks
from pdfminer.high_level import extract_text  # extract text from PDF


DOC_PATH = "./data/chain_of_thought_prompting.pdf"

# Suppose a user query
USER_QUERY = "What is CoT?"


# load local PDF document, you can use DirectoryLoader from langchain_community.document_loaders 
# if you have multiple documents
document_text = Document(extract_text(DOC_PATH))

# The document is long so it won't fit in the embeddings model context window
# We split it into chunks of 512 characters with 30 characters overlap
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=30)
chunked_docs = splitter.split_documents([document_text])

# you can choose an embedding model of your own
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# creates a db with the embeddings of the chunks for quick retrieval
db = InMemoryVectorStore.from_documents(chunked_docs, embeddings_model)

# Here we define how to search for similar chunks
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# Let's retrieve relevant chunks
retrieved_docs = retriever.invoke(USER_QUERY)
# let's format the docs into a context
context = [doc.page_content for doc in retrieved_docs]

conversation = [
    {"role": "system", "content": f"You are a helpful assistant. Reply to the user query considering the following context: [BEGIN_CONTEXT] {context[:2]} [END_CONTEXT]"},
    {"role": "user", "content": USER_QUERY},
]

# preprocess and generate same as before
text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
inputs = processor(text=[text_prompt], padding=True, return_tensors="pt")
inputs = inputs.to("cuda")
output_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(inputs.input_ids, output_ids)
]
output_text = processor.batch_decode(
    generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
)
print(output_text[0])

### 5. Create a user interface

#### Exercise 7

Since you are trying to build a complete application, you also need a nice user interface that interacts with the model. There are various libraries available for this purpose e.g., [gradio](https://www.gradio.app/docs/gradio/interface), [chat UI](https://huggingface.co/docs/chat-ui/index), [streamlit](https://docs.streamlit.io/) and more. For the solution of this lab, we will use gradio.

Gradio has pre-defined input/output blocks that are automatically inserted in the interface. You only need to provide an appropriate function that takes all the inputs and returns the relevant output. See documentation [here](https://www.gradio.app/docs/gradio/interface).

In [ ]:
import gradio as gr

# This part closes the demo server if it is already running (which
# happens easily in notebooks) and prevents you from opening multiple
# servers at the same time.
if "demo" in locals() and demo.is_running:
    demo.close()

# Same as before but images are taken from inputs and it also needs the history of the chat
def input2output_chatbot(inputs, history):
    conversation = []
    images = []

    # Let's restore the history
    suspended_images = 0  # to keep track of whether the history has any images to be inserted
    for conv in history:
        conversation.append(
            {
                "role": conv["role"],
                "content": [],
            },
        )
        for content in conv["content"]:
            if "text" in content:
                conversation[-1]["content"].append({"type": "text", "text": content["text"]})
            elif "file" in content:
                # confirm it's an image
                if "image" in content["file"]["mime_type"]:
                    image = Image.open(content["file"]["path"])
                    images.append(image)
                    conversation[-1]["content"].append({"type": "image"})
                    suspended_images += 1
                else:
                    print("Invalid file type in history.")

    # Now add user query to the conversation
    conversation.append(
        {
            "role": "user",
            "content": [{"type": "text", "text": inputs["text"]}],
        },
    )

    # check whether there are images in the current input
    if "files" in inputs:
        for image_dict in inputs["files"]:
            # check that path points to image
            if isinstance(image_dict, str):
                image = Image.open(image_dict)
            elif "image" in image_dict["mime_type"]:
                image = Image.open(image_dict["path"])
            else:
                raise ValueError("Invalid file type. Please upload an image.")
            conversation[-1]["content"].insert(0, {"type": "image"})
            images.append(image)

    # From now on is the same as before
    text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(
        text=[text_prompt],
        images=images if images else None,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)
    generated_ids = model.generate(**inputs, max_new_tokens=4096)
    generated_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return generated_text[0]


demo = gr.ChatInterface(input2output_chatbot, multimodal=True)
demo.launch()